# Intrinsic Evaluation — Gemma-2-9B

Environment Setup and Authentication

In [1]:
import sys
import os

# --- Work around a known huggingface_hub download hang ---
# The default XET download backend can deadlock on its own file lock in
# containerized environments like Kaggle (see huggingface/huggingface_hub#4508),
# leaving the from_pretrained() call stuck forever with no error or timeout.
# Disabling XET falls back to the plain HTTP downloader, which doesn't hang.
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_ETAG_TIMEOUT"] = "30"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "60"

# Clear any stale lock/incomplete files left over from a previous interrupted
# download -- these can make the very next attempt hang immediately.
!rm -rf ~/.cache/huggingface/hub/.locks
!find ~/.cache/huggingface/hub -name '*.incomplete' -delete 2>/dev/null || true

# Pin transformers to a stable release. Bleeding-edge 5.x releases have had
# regressions affecting several model configs (missing default attributes
# on load), so avoid a blind --upgrade to whatever is newest.
!{sys.executable} -m pip install -q "transformers==4.57.1" "accelerate>=0.26.0" "huggingface_hub>=0.25.0"

import torch
import math
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import login

# Hardware check
device_count = torch.cuda.device_count()
print(f"PyTorch {torch.__version__} | Detected {device_count} GPUs.")

import transformers as _tf
print(f"Transformers version: {_tf.__version__}")

# Authenticatation
# NOTE: Gemma-2-9B is a gated model on Hugging Face. Before running this
# notebook you must (1) accept the license at https://huggingface.co/google/gemma-2-9b
# with your HF account, and (2) make sure the HF_TOKEN Kaggle secret belongs
# to that same account -- otherwise the model download below will fail with
# a 401/403 "gated repo" error regardless of the fixes in this notebook.
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
    login(token=HF_TOKEN)
    print("Authenticated via Kaggle Secrets.")
except Exception as e:
    HF_TOKEN = ""
    login(token=HF_TOKEN)
    print("Authenticated via explicit string.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 84.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 31.7 MB/s eta 0:00:00
PyTorch 2.10.0+cu128 | Detected 2 GPUs.
Transformers version: 4.57.1
Authenticated via Kaggle Secrets.


Evaluation Metrices

In [2]:
class IntrinsicEvaluator:
    def __init__(self, model, tokenizer, device="cuda:0"):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device

    def get_lengths(self, text: str):
        """Calculates text-intrinsic properties for normalization."""
        if not isinstance(text, str):
            text = str(text)
        num_chars = max(1, len(text))
        num_bytes = max(1, len(text.encode('utf-8')))
        num_words = max(1, len(text.strip().split()))
        return num_chars, num_bytes, num_words

    def evaluate_sequence(self, text: str):
        """Calculates raw NLL plus the raw counts needed to pool correctly across the
        whole corpus. bpb/bpc/bpw below are still returned for per-sentence inspection,
        but the CORPUS-LEVEL (headline) numbers must be computed by pooling nll and the
        *_count fields across all rows -- never by averaging these per-row ratios."""
        chars, bytes_count, words = self.get_lengths(text)
        inputs = self.tokenizer(text, return_tensors="pt").to(self.device)

        with torch.no_grad():
            outputs = self.model(**inputs, labels=inputs["input_ids"])
            seq_len = inputs["input_ids"].shape[1]
            n_scored = seq_len - 1          # tokens actually predicted -- needed for token PPL
            neg_log_likelihood = outputs.loss.item() * n_scored

        bpb = neg_log_likelihood / (bytes_count * math.log(2))
        bpc = neg_log_likelihood / (chars * math.log(2))
        bpw = neg_log_likelihood / (words * math.log(2))

        return {
            "nll": neg_log_likelihood,
            "bpb": bpb, "bpc": bpc, "bpw": bpw,
            "n_scored": n_scored,
            "n_chars": chars, "n_bytes": bytes_count, "n_words": words,
        }
print("IntrinsicEvaluator class defined.")


IntrinsicEvaluator class defined.


Standardized Data Loading & Preprocessing

In [3]:
import glob

diverse_sentences_path = "/kaggle/input/datasets/anon-owner-e/eshinmenusha/diverse_sentences (1).csv"
sinhala_mixed_path = "/kaggle/input/datasets/anon-owner-e/eshinmenusha/sinhala_mixed_dataset.csv"

def resolve_path(preferred_path, filename_pattern):
    """Use the hard-coded path if it exists in this session; otherwise search
    /kaggle/input for a matching file. Datasets can be attached under a
    different owner/slug, or the exact filename can shift between versions
    (e.g. 'diverse_sentences.csv' vs 'diverse_sentences (1).csv'), which
    silently breaks a hard-coded path across sessions."""
    if os.path.exists(preferred_path):
        return preferred_path
    matches = glob.glob(f"/kaggle/input/**/{filename_pattern}", recursive=True)
    if matches:
        print(f"Note: preferred path not found, using discovered path instead: {matches[0]}")
        return matches[0]
    return preferred_path  # fall through so the original error message still fires

diverse_sentences_path = resolve_path(diverse_sentences_path, "diverse_sentences*.csv")
sinhala_mixed_path = resolve_path(sinhala_mixed_path, "sinhala_mixed_dataset*.csv")

try:
    diverse_df = pd.read_csv(diverse_sentences_path).dropna()
    mixed_df = pd.read_csv(sinhala_mixed_path).dropna()
    print(f"Loaded Diverse Sentences: {len(diverse_df)} parallel pairs.")
    print(f"Loaded Mixed Script: {len(mixed_df)} sentences.")
except Exception as e:
    print(f"Data loading failed. Check your file paths. Error: {e}")
    print("\nIs the 'anon-owner-e/eshinmenusha' dataset attached to this notebook?")
    print("Add it via the 'Add Input' panel on the right, or check Files below for what IS attached:")
    for p in sorted(glob.glob("/kaggle/input/*")):
        print(" -", p)


Loaded Diverse Sentences: 500 parallel pairs.
Loaded Mixed Script: 500 sentences.


Model & Tokenizer Initialization

In [4]:
from huggingface_hub import snapshot_download

model_id = "google/gemma-2-9b"
print(f"Loading {model_id}...")

tokenizer = AutoTokenizer.from_pretrained(model_id, token=HF_TOKEN, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Pre-download the weights explicitly (with XET disabled -- see setup cell) so
# any lock/network/gated-access issue surfaces here with a clear message,
# rather than inside from_pretrained() where it can look like a silent hang.
local_path = snapshot_download(
    repo_id=model_id,
    token=HF_TOKEN,
)
print(f"Weights cached at: {local_path}")

# Verify CUDA is actually usable for this specific GPU/PyTorch build --
# torch.cuda.is_available() can return True even when the GPU's compute
# capability isn't supported by the installed PyTorch wheel (e.g. older
# P100s on some Kaggle sessions), which would otherwise surface later as a
# confusing CUDA error mid-evaluation instead of at load time.
def get_usable_device():
    if not torch.cuda.is_available():
        return "cpu"
    try:
        torch.zeros(1, device="cuda") + 1
        return "cuda:0"
    except Exception as e:
        print(f"CUDA reported available but a test op failed ({e}); falling back to CPU.")
        return "cpu"

run_device = get_usable_device()
print(f"Using device: {run_device}")

# Load model natively in fp16 across available GPU(s), or fp32 on CPU
model = AutoModelForCausalLM.from_pretrained(
    local_path,
    device_map="auto" if run_device != "cpu" else None,
    dtype=torch.float16 if run_device != "cpu" else torch.float32,
    low_cpu_mem_usage=True,
    trust_remote_code=True
)
if run_device == "cpu":
    model = model.to("cpu")
model.eval()

# Initialize our custom evaluator with the device that was actually resolved
evaluator = IntrinsicEvaluator(model, tokenizer, device=run_device)

print(f"Model loaded.")
# hf_device_map is only set by transformers/accelerate when the model is
# dispatched across multiple devices -- it doesn't exist for a single-device
# load, so use getattr() instead of accessing it directly.
print(f"Precision: {model.dtype} | Device: {run_device} | Device Map: {getattr(model, 'hf_device_map', 'n/a (single device)')}")


Loading google/gemma-2-9b...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

(…)ransformers-4.42.0.dev0-py3-none-any.whl:   0%|          | 0.00/9.21M [00:00<?, ?B/s]

Weights cached at: /root/.cache/huggingface/hub/models--google--gemma-2-9b/snapshots/33c193028431c2fde6c6e51f29e6f17b60cbfac6
Using device: cuda:0


Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

Model loaded.
Precision: torch.float16 | Device: cuda:0 | Device Map: {'model.embed_tokens': 0, 'lm_head': 0, 'model.layers.0': 0, 'model.layers.1': 0, 'model.layers.2': 0, 'model.layers.3': 0, 'model.layers.4': 0, 'model.layers.5': 0, 'model.layers.6': 0, 'model.layers.7': 0, 'model.layers.8': 0, 'model.layers.9': 0, 'model.layers.10': 0, 'model.layers.11': 0, 'model.layers.12': 0, 'model.layers.13': 0, 'model.layers.14': 0, 'model.layers.15': 0, 'model.layers.16': 0, 'model.layers.17': 0, 'model.layers.18': 1, 'model.layers.19': 1, 'model.layers.20': 1, 'model.layers.21': 1, 'model.layers.22': 1, 'model.layers.23': 1, 'model.layers.24': 1, 'model.layers.25': 1, 'model.layers.26': 1, 'model.layers.27': 1, 'model.layers.28': 1, 'model.layers.29': 1, 'model.layers.30': 1, 'model.layers.31': 1, 'model.layers.32': 1, 'model.layers.33': 1, 'model.layers.34': 1, 'model.layers.35': 1, 'model.layers.36': 1, 'model.layers.37': 1, 'model.layers.38': 1, 'model.layers.39': 1, 'model.layers.40': 1

Pilot Run

In [5]:
import csv

pilot_output_path = 'pilot_intrinsic_results_gemma2_9b.csv'
print("Starting Pilot Run (First 20 items)...")

with open(pilot_output_path, mode='w', newline='', encoding='utf-8-sig') as f:
    writer = csv.writer(f)
    writer.writerow([
        'item_index',
        'unicode_nll', 'unicode_bpb', 'unicode_bpc', 'unicode_bpw',
        'unicode_nscored', 'unicode_nchars', 'unicode_nbytes', 'unicode_nwords',
        'romanized_nll', 'romanized_bpb', 'romanized_bpc', 'romanized_bpw',
        'romanized_nscored', 'romanized_nchars', 'romanized_nbytes', 'romanized_nwords'
    ])

    pilot_sample = diverse_df.head(20)

    for index, row in pilot_sample.iterrows():
        uni_metrics = evaluator.evaluate_sequence(row['sinhala_unicode'])
        rom_metrics = evaluator.evaluate_sequence(row['sinhala_romanized'])

        writer.writerow([
            index,
            uni_metrics['nll'], uni_metrics['bpb'], uni_metrics['bpc'], uni_metrics['bpw'],
            uni_metrics['n_scored'], uni_metrics['n_chars'], uni_metrics['n_bytes'], uni_metrics['n_words'],
            rom_metrics['nll'], rom_metrics['bpb'], rom_metrics['bpc'], rom_metrics['bpw'],
            rom_metrics['n_scored'], rom_metrics['n_chars'], rom_metrics['n_bytes'], rom_metrics['n_words']
        ])

        if (index + 1) % 5 == 0:
            print(f"Processed {index + 1}/20 parallel pairs...")

print(f"\n Pilot run complete. Results saved to {pilot_output_path}")
preview_df = pd.read_csv(pilot_output_path)
print("\nFirst 3 rows of calculated metrics:")
display(preview_df.head(3))


Starting Pilot Run (First 20 items)...
Processed 5/20 parallel pairs...
Processed 10/20 parallel pairs...
Processed 15/20 parallel pairs...
Processed 20/20 parallel pairs...

 Pilot run complete. Results saved to pilot_intrinsic_results_gemma2_9b.csv

First 3 rows of calculated metrics:


,item_index,unicode_nll,unicode_bpb,unicode_bpc,unicode_bpw,unicode_nscored,unicode_nchars,unicode_nbytes,unicode_nwords,romanized_nll,romanized_bpb,romanized_bpc,romanized_bpw,romanized_nscored,romanized_nchars,romanized_nbytes,romanized_nwords
0,0,66.631899,1.232430,3.204317,16.021585,26,30,78,6,102.388030,3.887229,3.887229,24.619117,13,38,38,6
1,1,47.090507,1.029352,2.830718,16.984310,20,24,66,4,93.318652,4.808227,4.808227,33.657589,9,28,28,4
2,2,55.910862,0.971835,2.602010,13.443721,22,31,83,6,93.402145,3.743078,3.743078,22.458469,11,36,36,6


Full Intrinsic Evaluation

In [6]:
import csv
import pandas as pd
from tqdm import tqdm

full_diverse_output = 'gemma2_9b_diverse_intrinsic.csv'
print(f"Starting full parallel evaluation on {len(diverse_df)} items...")

with open(full_diverse_output, mode='w', newline='', encoding='utf-8-sig') as f:
    writer = csv.writer(f)
    writer.writerow([
        'item_index',
        'unicode_nll', 'unicode_bpb', 'unicode_bpc', 'unicode_bpw',
        'unicode_nscored', 'unicode_nchars', 'unicode_nbytes', 'unicode_nwords',
        'romanized_nll', 'romanized_bpb', 'romanized_bpc', 'romanized_bpw',
        'romanized_nscored', 'romanized_nchars', 'romanized_nbytes', 'romanized_nwords'
    ])

    for index, row in tqdm(diverse_df.iterrows(), total=len(diverse_df), desc="Processing Parallel Pairs"):
        uni_metrics = evaluator.evaluate_sequence(row['sinhala_unicode'])
        rom_metrics = evaluator.evaluate_sequence(row['sinhala_romanized'])

        writer.writerow([
            index,
            uni_metrics['nll'], uni_metrics['bpb'], uni_metrics['bpc'], uni_metrics['bpw'],
            uni_metrics['n_scored'], uni_metrics['n_chars'], uni_metrics['n_bytes'], uni_metrics['n_words'],
            rom_metrics['nll'], rom_metrics['bpb'], rom_metrics['bpc'], rom_metrics['bpw'],
            rom_metrics['n_scored'], rom_metrics['n_chars'], rom_metrics['n_bytes'], rom_metrics['n_words']
        ])

print(f"Diverse evaluation complete. Saved to {full_diverse_output}")

full_mixed_output = 'gemma2_9b_mixed_intrinsic.csv'
mixed_text_col = 'text' if 'text' in mixed_df.columns else mixed_df.columns[0]
print(f"\nStarting full mixed-script evaluation on {len(mixed_df)} items (column: '{mixed_text_col}')...")

with open(full_mixed_output, mode='w', newline='', encoding='utf-8-sig') as f:
    writer = csv.writer(f)
    writer.writerow([
        'item_index', 'mixed_nll', 'mixed_bpb', 'mixed_bpc', 'mixed_bpw',
        'mixed_nscored', 'mixed_nchars', 'mixed_nbytes', 'mixed_nwords'
    ])

    for index, row in tqdm(mixed_df.iterrows(), total=len(mixed_df), desc="Processing Mixed Script"):
        mix_metrics = evaluator.evaluate_sequence(row[mixed_text_col])
        writer.writerow([
            index,
            mix_metrics['nll'], mix_metrics['bpb'], mix_metrics['bpc'], mix_metrics['bpw'],
            mix_metrics['n_scored'], mix_metrics['n_chars'], mix_metrics['n_bytes'], mix_metrics['n_words']
        ])

print(f"Mixed-script evaluation complete. Saved to {full_mixed_output}")


# --- Corrected aggregation: pooled (corpus-level) BPB/BPC/BPW + token-level Perplexity ---
def calculate_full_aggregates(csv_path, model_label="Gemma-2-9B"):
    df = pd.read_csv(csv_path)
    ln2 = math.log(2)

    def pooled_stats(prefix):
        nll_sum = df[f'{prefix}_nll'].sum()
        return {
            'bpb': nll_sum / (df[f'{prefix}_nbytes'].sum() * ln2),
            'bpc': nll_sum / (df[f'{prefix}_nchars'].sum() * ln2),
            'bpw': nll_sum / (df[f'{prefix}_nwords'].sum() * ln2),
            # this is the number to compare against the paper's Table:
            'ppl_token': math.exp(nll_sum / df[f'{prefix}_nscored'].sum()),
        }

    uni = pooled_stats('unicode')
    rom = pooled_stats('romanized')

    results_df = pd.DataFrame({
        'Metric': ['Bits-Per-Byte (BPB)', 'Bits-Per-Character (BPC)', 'Bits-Per-Word (BPW)',
                   'Token-Level Perplexity'],
        'Unicode (Native)': [uni['bpb'], uni['bpc'], uni['bpw'], uni['ppl_token']],
        'Romanized (Singlish)': [rom['bpb'], rom['bpc'], rom['bpw'], rom['ppl_token']],
    })
    results_df['Degradation Factor'] = results_df['Romanized (Singlish)'] / results_df['Unicode (Native)']

    print("==========================================================")
    print(f"      {model_label} INTRINSIC EVALUATION SUMMARY (corpus-pooled)")
    print("==========================================================")
    print(results_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    print("==========================================================")

    styled = results_df.style.format({
        'Unicode (Native)': '{:.4f}',
        'Romanized (Singlish)': '{:.4f}',
        'Degradation Factor': '{:.2f}×'
    }).set_caption(f"{model_label} — Unicode vs Romanized Sinhala (corpus-pooled)").background_gradient(
        subset=['Degradation Factor'], cmap='Reds'
    )
    display(styled)

    return results_df

full_summary = calculate_full_aggregates(full_diverse_output)


Starting full parallel evaluation on 500 items...


Processing Parallel Pairs: 100%|██████████| 500/500 [01:57<00:00,  4.27it/s]


Diverse evaluation complete. Saved to gemma2_9b_diverse_intrinsic.csv

Starting full mixed-script evaluation on 500 items (column: 'text')...


Processing Mixed Script: 100%|██████████| 500/500 [01:02<00:00,  7.97it/s]


Mixed-script evaluation complete. Saved to gemma2_9b_mixed_intrinsic.csv
      Gemma-2-9B INTRINSIC EVALUATION SUMMARY (corpus-pooled)
                  Metric  Unicode (Native)  Romanized (Singlish)  Degradation Factor
     Bits-Per-Byte (BPB)            1.1900                3.9986              3.3601
Bits-Per-Character (BPC)            3.1547                4.0098              1.2710
     Bits-Per-Word (BPW)           16.9200               24.6783              1.4585
  Token-Level Perplexity           13.2327             4726.0715            357.1517


,Metric,Unicode (Native),Romanized (Singlish),Degradation Factor
0,Bits-Per-Byte (BPB),1.1900,3.9986,3.36×
1,Bits-Per-Character (BPC),3.1547,4.0098,1.27×
2,Bits-Per-Word (BPW),16.9200,24.6783,1.46×
3,Token-Level Perplexity,13.2327,4726.0715,357.15×


In [7]:
def calculate_mixed_aggregates(csv_path, model_label="Gemma-2-9B"):
    df = pd.read_csv(csv_path)
    ln2 = math.log(2)
    nll_sum = df['mixed_nll'].sum()

    pooled_bpb = nll_sum / (df['mixed_nbytes'].sum() * ln2)
    pooled_bpc = nll_sum / (df['mixed_nchars'].sum() * ln2)
    pooled_bpw = nll_sum / (df['mixed_nwords'].sum() * ln2)
    pooled_ppl = math.exp(nll_sum / df['mixed_nscored'].sum())

    # per-sentence values, used ONLY for the dispersion column below -- not the headline numbers above
    per_sentence_ppl = (df['mixed_nll'] / df['mixed_nscored']).apply(math.exp)

    results_df = pd.DataFrame({
        'Metric': ['Bits-Per-Byte (BPB)', 'Bits-Per-Character (BPC)', 'Bits-Per-Word (BPW)',
                   'Token-Level Perplexity'],
        'Mixed-Script (corpus-pooled)': [pooled_bpb, pooled_bpc, pooled_bpw, pooled_ppl],
        'Per-sentence Std Dev': [df['mixed_bpb'].std(), df['mixed_bpc'].std(),
                                 df['mixed_bpw'].std(), per_sentence_ppl.std()],
    })

    print("==========================================================")
    print(f"      {model_label} MIXED-SCRIPT INTRINSIC SUMMARY")
    print(f"      ({len(df)} sentences)")
    print("==========================================================")
    print(results_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    print("==========================================================")

    styled = results_df.style.format({
        'Mixed-Script (corpus-pooled)': '{:.4f}',
        'Per-sentence Std Dev': '{:.4f}'
    }).set_caption(f"{model_label} — Mixed-Script Sinhala")
    display(styled)

    return results_df

mixed_summary = calculate_mixed_aggregates(full_mixed_output)


      Gemma-2-9B MIXED-SCRIPT INTRINSIC SUMMARY
      (500 sentences)
                  Metric  Mixed-Script (corpus-pooled)  Per-sentence Std Dev
     Bits-Per-Byte (BPB)                        1.7923                0.6192
Bits-Per-Character (BPC)                        3.5328                0.5989
     Bits-Per-Word (BPW)                       21.4660                4.4352
  Token-Level Perplexity                       48.9923              628.8294


,Metric,Mixed-Script (corpus-pooled),Per-sentence Std Dev
0,Bits-Per-Byte (BPB),1.7923,0.6192
1,Bits-Per-Character (BPC),3.5328,0.5989
2,Bits-Per-Word (BPW),21.4660,4.4352
3,Token-Level Perplexity,48.9923,628.8294
